# Bulk Download: Time Series for One Bank

Goal: pull Call Report XBRL data for one bank across several quarters, combine it into a single DataFrame, and plot a metric over time.

**Use case:** You're tracking how a bank's total assets, loans, or deposits have evolved quarter-over-quarter.

**Tip:** The library's built-in rate limiter (2500 req/hr) means you can safely pull dozens of quarters in sequence without worrying about throttling.

## Step 0: Dev setup, environment check & credentials

This cell does three things:

1. **Dev toggle** — set `USE_LOCAL_SRC = True` to import the library directly from this repo's `src/` tree instead of a pip-installed version. Useful for developers testing changes, or for trying the library without installing it. Regular users leave it `False`.
2. **Environment check** — verifies Python >= 3.11 and all required dependencies, so you get a clear error up front instead of a confusing failure mid-notebook.
3. **Credentials helpers** — defines two functions you can call in Step 1:
    - `show_credentials_form()` — renders an ipywidgets form (username + password field + Submit button) right in the notebook. Best for interactive use.
    - `get_credentials()` — blocking text-prompt fallback. Best for automation / scripted notebooks.
    Both first check `FFIEC_USERNAME` / `FFIEC_BEARER_TOKEN` env vars and a `.env` file, and skip the UI if those already supply both values.

**Three ways to supply credentials** (Step 0 picks whichever is available):

- **Env vars** — `export FFIEC_USERNAME=... FFIEC_BEARER_TOKEN=ey...` before launching Jupyter.
- **`.env` file** — create `.env` in the notebook's directory with the same two variables. Requires `pip install python-dotenv`. Remember to `.gitignore` it.
- **Interactive form** — leave both unset and `show_credentials_form()` (in Step 1) will render a widget with a hidden password field. Requires `pip install ipywidgets` (falls back to a text prompt otherwise).

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# Flip to True to load ffiec_data_connect from the local repo src/ tree (dev mode).
USE_LOCAL_SRC = False

if USE_LOCAL_SRC:
    # Walk upward from the notebook's CWD to find a sibling `src/ffiec_data_connect`.
    here = Path.cwd().resolve()
    src_root = None
    for candidate in [here, *here.parents]:
        if (candidate / 'src' / 'ffiec_data_connect' / '__init__.py').exists():
            src_root = candidate / 'src'
            break
    if src_root is None:
        raise RuntimeError(
            'USE_LOCAL_SRC=True but no src/ffiec_data_connect found '
            'walking up from ' + str(here)
        )
    sys.path.insert(0, str(src_root))
    # Evict any previously-imported copy so the src/ tree wins on next import.
    for mod_name in [m for m in sys.modules if m == 'ffiec_data_connect' or m.startswith('ffiec_data_connect.')]:
        del sys.modules[mod_name]
    print(f'Loading ffiec_data_connect from: {src_root}')

# --- Environment support check -------------------------------------------
if sys.version_info < (3, 11):
    raise RuntimeError(
        f'ffiec-data-connect requires Python >= 3.11, got {sys.version.split()[0]}'
    )

REQUIRED = ['httpx', 'pandas', 'numpy', 'lxml', 'xmltodict', 'defusedxml', 'pydantic']
missing = []
for mod in REQUIRED:
    try:
        __import__(mod)
    except ImportError:
        missing.append(mod)
if missing:
    raise RuntimeError(
        'Missing required dependencies: ' + ', '.join(missing) +
        '\nInstall with: pip install ffiec-data-connect'
    )

# Optional deps used by some notebooks — warn but don't fail here.
OPTIONAL = {
    'matplotlib': 'plotting cells (notebooks 02)',
    'polars': 'polars output_type (optional extra: pip install ffiec-data-connect[polars])',
    'pyarrow': 'parquet I/O (pip install pyarrow)',
    'ipywidgets': 'interactive credentials form via show_credentials_form() (pip install ipywidgets)',
    'dotenv': '.env file support for credentials (pip install python-dotenv)',
}
for mod, purpose in OPTIONAL.items():
    try:
        __import__(mod)
    except ImportError:
        warnings.warn(f'Optional dep {mod!r} not installed — needed for {purpose}', stacklevel=1)

import ffiec_data_connect
from ffiec_data_connect import OAuth2Credentials
print(f'Python:              {sys.version.split()[0]}')
print(f'ffiec_data_connect:  {ffiec_data_connect.__version__}')
print(f'Loaded from:         {Path(ffiec_data_connect.__file__).parent}')
print('Environment OK.')

# --- Credentials helpers -------------------------------------------------
# Two ways to authenticate:
#   get_credentials()       — blocking. Tries env vars, then .env, then falls back to text prompts (input/getpass). Best for automation / scripts.
#   show_credentials_form() — renders an ipywidgets form (username field + password field + Submit button). Best for interactive notebook use. Sets `creds` in the notebook's namespace when you click Submit.

def _resolve_env_credentials():
    """Return (username, token) from env vars / .env, else (None|str, None|str)."""
    username = os.environ.get('FFIEC_USERNAME')
    token = os.environ.get('FFIEC_BEARER_TOKEN')
    if not (username and token):
        try:
            from dotenv import load_dotenv
            load_dotenv()
            username = username or os.environ.get('FFIEC_USERNAME')
            token = token or os.environ.get('FFIEC_BEARER_TOKEN')
        except ImportError:
            pass
    return username, token

def _warn_about_token_shape(token: str) -> None:
    """Soft-warn about common token-paste mistakes before OAuth2Credentials raises hard.

    FFIEC-issued JWT bearer tokens have an idiosyncratic shape: they start with 'ey' and
    **end with a literal '.'** (an empty third segment). Browsers and copy-paste sometimes
    strip the trailing '.', which silently breaks auth. We warn early so a user can re-copy
    rather than hitting a validation exception deeper in the call.
    """
    if not token:
        return
    t = token.strip()
    if not t.endswith('.'):
        warnings.warn(
            'Bearer token does not end with a ".". FFIEC JWT tokens are issued with a trailing '
            '"." (empty third segment) — a missing dot usually means the paste was truncated. '
            'Re-copy from the Account Details tab at https://cdr.ffiec.gov/public/PWS/PublicLogin.aspx',
            stacklevel=2,
        )
    if not t.startswith('ey'):
        warnings.warn(
            "Bearer token does not start with 'ey'. FFIEC JWTs begin with the base64url header "
            "'ey...' — this token likely isn't a JWT.",
            stacklevel=2,
        )

def get_credentials():
    """Blocking credential fetch: env vars -> .env -> text prompts. Returns OAuth2Credentials."""
    username, token = _resolve_env_credentials()
    if not username:
        username = input('FFIEC username: ').strip()
    if not token:
        import getpass
        token = getpass.getpass('FFIEC bearer token (hidden): ').strip()
    _warn_about_token_shape(token)
    creds = OAuth2Credentials(username=username, bearer_token=token)
    print(f'Credentials loaded. Token expires: {creds.token_expires} (expired? {creds.is_expired})')
    return creds

def show_credentials_form(target_name: str = 'creds'):
    """Render an ipywidgets credentials form. On Submit, sets globals()[target_name] to an OAuth2Credentials.

    If ipywidgets / IPython aren't available, falls through to get_credentials() (blocking text prompts).
    If env vars or .env already supply both values, skips the form entirely and returns creds directly.
    """
    # Fast path: env/.env already satisfied both fields — no UI needed.
    username, token = _resolve_env_credentials()
    if username and token:
        _warn_about_token_shape(token)
        creds = OAuth2Credentials(username=username, bearer_token=token)
        print(f'Credentials loaded from environment. Token expires: {creds.token_expires} (expired? {creds.is_expired})')
        # Inject into caller's notebook globals so downstream cells see `creds`.
        import inspect
        caller_globals = inspect.stack()[1].frame.f_globals
        caller_globals[target_name] = creds
        return creds

    # Try the widget path; fall back to text prompts if anything is missing.
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError:
        print('ipywidgets not available — falling back to text prompts.')
        print('  (pip install ipywidgets for a nicer form-based input)')
        creds = get_credentials()
        import inspect
        inspect.stack()[1].frame.f_globals[target_name] = creds
        return creds

    # Capture the caller's globals so the Submit handler can write `creds` back.
    import inspect
    caller_globals = inspect.stack()[1].frame.f_globals

    user_field = widgets.Text(
        value=username or '',
        placeholder='FFIEC PWS username',
        description='Username:',
        layout=widgets.Layout(width='420px'),
    )
    token_field = widgets.Password(
        value=token or '',
        placeholder='JWT bearer token (starts with ey..., ends with .)',
        description='Token:',
        layout=widgets.Layout(width='420px'),
    )
    submit = widgets.Button(description='Submit', button_style='primary', icon='check')
    status = widgets.Output()

    def _on_submit(_):
        status.clear_output()
        with status:
            u = (user_field.value or '').strip()
            t = (token_field.value or '').strip()
            if not u or not t:
                print('Please fill in both fields.')
                return
            # Surface paste-truncation warnings visibly in the form area.
            if not t.endswith('.'):
                print('⚠ Warning: token does not end with ".". FFIEC JWTs end in a literal "." — '
                      'a missing dot usually means the paste was truncated. Re-copy if auth fails.')
            if not t.startswith('ey'):
                print("⚠ Warning: token does not start with 'ey' — likely not a JWT.")
            try:
                creds = OAuth2Credentials(username=u, bearer_token=t)
            except Exception as e:
                print(f'Invalid credentials: {e}')
                return
            caller_globals[target_name] = creds
            print(f'✓ Credentials set in `{target_name}`. Token expires: {creds.token_expires} (expired? {creds.is_expired}).')
            print('  Proceed to the next cell.')

    submit.on_click(_on_submit)
    display(widgets.VBox([user_field, token_field, submit, status]))
    print('Fill the form above and click Submit. `' + target_name + '` will be set in your notebook.')
    return None  # Value is delivered via globals[target_name] on Submit.

print('Two ways to authenticate in the next cell:')
print('  creds = get_credentials()       # blocking, text prompts')
print('  show_credentials_form()         # interactive ipywidgets form (sets `creds` on Submit)')

## Setup

In [ ]:
import pandas as pd
from ffiec_data_connect import (
    collect_data,
    collect_reporting_periods,
)

### Authenticate

Running the next cell shows an interactive form unless env vars / `.env` already provide credentials. Submitting sets `creds` in the notebook namespace. For scripted use, replace with `creds = get_credentials()`.

In [ ]:
show_credentials_form()   # Sets `creds` on Submit.

## Pick a bank and a date range

For this example: Bank of America (RSSD 480228), last 8 quarters.

In [ ]:
RSSD_ID = '480228'  # Bank of America, N.A.
NUM_QUARTERS = 8

all_periods = collect_reporting_periods(creds, series='call')
# Skip the latest period — it may be a future quarter not yet filed
target_periods = all_periods[-(NUM_QUARTERS + 1):-1]

print(f'Fetching {len(target_periods)} quarters:')
for p in target_periods:
    print(f'  {p}')

## Fetch each quarter

Each call returns the full XBRL filing. We tag each row with its reporting period so we can concatenate into a long-format DataFrame.

In [ ]:
quarterly_data = []

for period in target_periods:
    print(f'Fetching {period}...', end=' ')
    df = collect_data(
        creds,
        reporting_period=period,
        rssd_id=RSSD_ID,
        series='call',
        output_type='pandas',
    )
    df['reporting_period'] = period
    quarterly_data.append(df)
    print(f'{len(df)} items')

combined = pd.concat(quarterly_data, ignore_index=True)
print(f'\nCombined: {len(combined):,} rows')

## Extract one metric across time

Call Report data uses MDRM codes to identify individual data points. For example:
- `RCFD2170` — Total assets (consolidated)
- `RCFD2200` — Total deposits
- `RCFD2122` — Total loans and leases, net

See the [Call Report MDRM Data Dictionary](https://cdr.ffiec.gov/public/ManageFacsimiles.aspx) for the full list.

In [ ]:
# Extract total assets over time.
# RCFD = consolidated (banks with foreign offices); RCON = domestic-only. Bank of America
# files RCFD. For a small community bank, swap to 'RCON2170' or accept either via .isin().
MDRM_TOTAL_ASSETS_CODES = ['RCFD2170', 'RCON2170']

time_series = (
    combined[combined['mdrm'].isin(MDRM_TOTAL_ASSETS_CODES)]
    [['reporting_period', 'int_data']]
    .rename(columns={'int_data': 'total_assets_thousands'})
    .sort_values('reporting_period')
    .reset_index(drop=True)
)

if time_series.empty:
    raise ValueError(
        f'No rows matched {MDRM_TOTAL_ASSETS_CODES} in the combined data. '
        f'Did the fetch loop succeed? Sample of available MDRMs: '
        f'{combined["mdrm"].value_counts().head(10).to_dict()}'
    )

time_series

## Plot it

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(time_series['reporting_period'], time_series['total_assets_thousands'] / 1e6, marker='o')
ax.set_title(f'Total Assets Over Time — RSSD {RSSD_ID}')
ax.set_xlabel('Reporting Period')
ax.set_ylabel('Total Assets ($B)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Save for later analysis

For production pipelines, save the combined DataFrame to Parquet — efficient, preserves dtypes, and fast to reload.

In [ ]:
combined.to_parquet('bank_480228_8q.parquet', index=False)
print(f'Saved {len(combined):,} rows')